In [1]:
from google.colab import drive

drive.mount("/content/gdrive")

Mounted at /content/gdrive


EVID-01 — Initialize notebook

In [2]:
import os
import json
import pandas as pd
from datetime import datetime

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

EVIDENCE_RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Evidence"
)

os.makedirs(EVIDENCE_RESULTS_DIR, exist_ok=True)

print("CrisisWeave Evidence Construction initialized.")
print("Output directory:", EVIDENCE_RESULTS_DIR)

CrisisWeave Evidence Construction initialized.
Output directory: /content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Evidence


EVID-02 — Load GIS and Temporal outputs

In [3]:
# Load saved GIS result
gis_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "GIS",
    "CrisisWeave_GIS_CW001.json"
)

with open(gis_path, "r", encoding="utf-8") as f:
    gis_output = json.load(f)


# Load saved Temporal result
temporal_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Temporal",
    "CrisisWeave_Temporal_State_CW001.json"
)

with open(temporal_path, "r", encoding="utf-8") as f:
    temporal_state = json.load(f)


print("GIS output loaded.")
print("Temporal output loaded.")

print("\nGIS keys:")
print(gis_output.keys())

print("\nTemporal keys:")
print(temporal_state.keys())

GIS output loaded.
Temporal output loaded.

GIS keys:
dict_keys(['incident_id', 'incident_location', 'gis_context', 'location_verification', 'candidate_locations'])

Temporal keys:
dict_keys(['incident_id', 'observation_count', 'first_observation', 'latest_observation', 'initial_severity', 'latest_severity', 'severity_trend', 'water_extent_trend', 'affected_population_trend', 'overall_temporal_trend', 'progression_direction', 'latest_latitude', 'latest_longitude'])


EVID-03 — Define the evidence schema

In [4]:
evidence_schema = {
    "incident_id": None,
    "evidence_id": None,

    "timestamp": None,

    "location": {
        "latitude": None,
        "longitude": None,
        "source": None
    },

    "image_evidence": {
        "image_path": None,
        "cnn_severity": None,
        "cnn_confidence": None
    },

    "text_evidence": {
        "text": None,
        "bert_category": None,
        "bert_confidence": None
    },

    "gis_evidence": {
        "nearest_road": None,
        "nearest_school": None,
        "nearest_hospital": None,
        "nearest_bridge": None,
        "nearest_waterbody": None,
        "water_relationship": None
    },

    "temporal_evidence": {
        "severity_trend": None,
        "water_extent_trend": None,
        "affected_population_trend": None,
        "overall_temporal_trend": None,
        "progression_direction": None
    },

    "response_evidence": {
        "response_status": None,
        "response_timestamp": None
    }
}

print(json.dumps(evidence_schema, indent=2))

{
  "incident_id": null,
  "evidence_id": null,
  "timestamp": null,
  "location": {
    "latitude": null,
    "longitude": null,
    "source": null
  },
  "image_evidence": {
    "image_path": null,
    "cnn_severity": null,
    "cnn_confidence": null
  },
  "text_evidence": {
    "text": null,
    "bert_category": null,
    "bert_confidence": null
  },
  "gis_evidence": {
    "nearest_road": null,
    "nearest_school": null,
    "nearest_hospital": null,
    "nearest_bridge": null,
    "nearest_waterbody": null,
    "water_relationship": null
  },
  "temporal_evidence": {
    "severity_trend": null,
    "water_extent_trend": null,
    "affected_population_trend": null,
    "overall_temporal_trend": null,
    "progression_direction": null
  },
  "response_evidence": {
    "response_status": null,
    "response_timestamp": null
  }
}


EVID-05 — Add source metadata

In [6]:
# EVID-05 — Recreate evidence record + add source metadata

import os
import json

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

# Load GIS result
gis_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "GIS",
    "CrisisWeave_GIS_CW001.json"
)

with open(gis_path, "r", encoding="utf-8") as f:
    gis_output = json.load(f)


# Load Temporal result
temporal_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Temporal",
    "CrisisWeave_Temporal_State_CW001.json"
)

with open(temporal_path, "r", encoding="utf-8") as f:
    temporal_state = json.load(f)


# Recreate unified evidence record
evidence_record = {
    "incident_id": "CW-001",
    "evidence_id": "EV-001",

    "timestamp": temporal_state["latest_observation"],

    "location": {
        "latitude": gis_output["incident_location"]["latitude"],
        "longitude": gis_output["incident_location"]["longitude"],
        "source": "user_map_pin"
    },

    "image_evidence": {
        "image_path": None,
        "cnn_severity": "High",
        "cnn_confidence": None
    },

    "text_evidence": {
        "text": "Water has entered the road and several people need help.",
        "bert_category": "requests_or_urgent_needs",
        "bert_confidence": None
    },

    "gis_evidence": {
        "nearest_road": gis_output["gis_context"].get("nearest_road"),
        "nearest_school": gis_output["gis_context"].get("nearest_school"),
        "nearest_hospital": gis_output["gis_context"].get("nearest_hospital"),
        "nearest_bridge": gis_output["gis_context"].get("nearest_bridge"),
        "nearest_waterbody": gis_output["gis_context"].get("nearest_waterbody"),
        "water_relationship": gis_output["gis_context"].get("water_relationship")
    },

    "temporal_evidence": {
        "severity_trend": temporal_state["severity_trend"],
        "water_extent_trend": temporal_state["water_extent_trend"],
        "affected_population_trend": temporal_state["affected_population_trend"],
        "overall_temporal_trend": temporal_state["overall_temporal_trend"],
        "progression_direction": temporal_state["progression_direction"]
    },

    "response_evidence": {
        "response_status": "En Route",
        "response_timestamp": temporal_state["latest_observation"]
    },

    "source_metadata": {
        "image_source": "CNN",
        "text_source": "BERT",
        "gis_source": "OpenStreetMap",
        "temporal_source": "Temporal_Analysis",
        "response_source": "Response_Event"
    }
}

print("Evidence record recreated successfully.\n")
print(json.dumps(
    evidence_record,
    indent=2,
    ensure_ascii=False,
    default=str
))

Evidence record recreated successfully.

{
  "incident_id": "CW-001",
  "evidence_id": "EV-001",
  "timestamp": "2026-09-11T12:00:00",
  "location": {
    "latitude": 9.929117,
    "longitude": 76.270436,
    "source": "user_map_pin"
  },
  "image_evidence": {
    "image_path": null,
    "cnn_severity": "High",
    "cnn_confidence": null
  },
  "text_evidence": {
    "text": "Water has entered the road and several people need help.",
    "bert_category": "requests_or_urgent_needs",
    "bert_confidence": null
  },
  "gis_evidence": {
    "nearest_road": {
      "feature_type": "Road",
      "name": "Syrian Church Road",
      "latitude": 1097835.502269,
      "longitude": 639258.587067,
      "distance_m": 17.2
    },
    "nearest_school": {
      "feature_type": "School",
      "name": "Kendriya Vidyalaya-Mundamveli Kochi",
      "latitude": 1098032.308252,
      "longitude": 638927.3767,
      "distance_m": 358.26
    },
    "nearest_hospital": {
      "feature_type": "Hospital",
   

EVID-06 — Save unified evidence record

In [7]:
# EVID-06 — Save unified evidence record

EVIDENCE_RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Evidence"
)

os.makedirs(EVIDENCE_RESULTS_DIR, exist_ok=True)

evidence_json_path = os.path.join(
    EVIDENCE_RESULTS_DIR,
    "CrisisWeave_Evidence_CW001_EV001.json"
)

with open(
    evidence_json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        evidence_record,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )

print("Unified evidence record saved:")
print(evidence_json_path)

Unified evidence record saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Evidence/CrisisWeave_Evidence_CW001_EV001.json


EVID-07 — Create compact evidence table

In [8]:
# EVID-07 — Create compact evidence table

import os
import json
import pandas as pd

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

evidence_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Evidence",
    "CrisisWeave_Evidence_CW001_EV001.json"
)

# Load the saved evidence record
with open(evidence_path, "r", encoding="utf-8") as f:
    evidence_record = json.load(f)

evidence_table = pd.DataFrame([{
    "incident_id": evidence_record["incident_id"],
    "evidence_id": evidence_record["evidence_id"],
    "timestamp": evidence_record["timestamp"],

    "latitude": evidence_record["location"]["latitude"],
    "longitude": evidence_record["location"]["longitude"],

    "cnn_severity": evidence_record["image_evidence"]["cnn_severity"],
    "bert_category": evidence_record["text_evidence"]["bert_category"],

    "water_relationship":
        evidence_record["gis_evidence"]["water_relationship"],

    "severity_trend":
        evidence_record["temporal_evidence"]["severity_trend"],

    "water_extent_trend":
        evidence_record["temporal_evidence"]["water_extent_trend"],

    "affected_population_trend":
        evidence_record["temporal_evidence"]["affected_population_trend"],

    "overall_temporal_trend":
        evidence_record["temporal_evidence"]["overall_temporal_trend"],

    "progression_direction":
        evidence_record["temporal_evidence"]["progression_direction"],

    "response_status":
        evidence_record["response_evidence"]["response_status"]
}])

print("Unified evidence table:")
display(evidence_table)

Unified evidence table:


,incident_id,evidence_id,timestamp,latitude,longitude,cnn_severity,bert_category,water_relationship,severity_trend,water_extent_trend,affected_population_trend,overall_temporal_trend,progression_direction,response_status
0,CW-001,EV-001,2026-09-11T12:00:00,9.929117,76.270436,High,requests_or_urgent_needs,None,INCREASING,INCREASING,INCREASING,INCREASING,NE,En Route


EVID-08 — Save the table

In [9]:
# EVID-08 — Save compact evidence table

evidence_csv_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Evidence",
    "CrisisWeave_Evidence_CW001_EV001.csv"
)

evidence_table.to_csv(
    evidence_csv_path,
    index=False
)

print("Evidence table saved:")
print(evidence_csv_path)

Evidence table saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Evidence/CrisisWeave_Evidence_CW001_EV001.csv


EVID-09 — Add source reliability placeholders

In [10]:
# EVID-09 — Define source reliability metadata

source_reliability = {
    "CNN": {
        "source_type": "image_classifier",
        "reliability": None,
        "status": "PENDING"
    },

    "BERT": {
        "source_type": "text_classifier",
        "reliability": None,
        "status": "PENDING"
    },

    "OpenStreetMap": {
        "source_type": "geospatial_context",
        "reliability": None,
        "status": "PENDING"
    },

    "Temporal_Analysis": {
        "source_type": "temporal_evidence",
        "reliability": None,
        "status": "PENDING"
    },

    "Response_Event": {
        "source_type": "response_information",
        "reliability": None,
        "status": "PENDING"
    }
}

evidence_record["source_reliability"] = source_reliability

print(json.dumps(
    source_reliability,
    indent=2
))

{
  "CNN": {
    "source_type": "image_classifier",
    "reliability": null,
    "status": "PENDING"
  },
  "BERT": {
    "source_type": "text_classifier",
    "reliability": null,
    "status": "PENDING"
  },
  "OpenStreetMap": {
    "source_type": "geospatial_context",
    "reliability": null,
    "status": "PENDING"
  },
  "Temporal_Analysis": {
    "source_type": "temporal_evidence",
    "reliability": null,
    "status": "PENDING"
  },
  "Response_Event": {
    "source_type": "response_information",
    "reliability": null,
    "status": "PENDING"
  }
}


EVID-10 — Save final evidence package

In [11]:
# EVID-10 — Save evidence package with source metadata

final_evidence_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Evidence",
    "CrisisWeave_Evidence_CW001_EV001.json"
)

with open(
    final_evidence_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        evidence_record,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Final evidence package saved:")
print(final_evidence_path)

Final evidence package saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Evidence/CrisisWeave_Evidence_CW001_EV001.json
